# LDLR coding-variant splice site usage scoring (AlphaGenome)

Scores the predicted change in splice site usage for missense coding variants in LDLR, specified by amino-acid change (e.g. `T713G`, `Q770F`).

For each variant:
1. Look up the codon position in the LDLR CDS and map to genomic (chr19) coordinates.
2. Build the full alternate sequence with all codon substitutions applied simultaneously.
3. Score predicted splice site usage change by comparing `predict_sequence` on ref vs. alt, with a center-masked mean-delta identical to `CenterMaskScorer`.

This approach is uniform for all variants regardless of how many nucleotide positions change within the codon — there is no additivity assumption and epistatic interactions between co-occurring substitutions are captured correctly.

In [10]:
!pip install alphagenome
from inspect import signature
from alphagenome import colab_utils
from alphagenome.data import genome, gene_annotation
from alphagenome.models import dna_client
from alphagenome.data import transcript as transcript_utils
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.interpretation import ism as ism_utils
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd
import os

# Set your API key as an environment variable before launching jupyter:
#   export ALPHA_GENOME_API_KEY='your-key-here'
API_KEY = 'YOUR KEY HERE'
model = dna_client.create(API_KEY)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.9/176.9 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 58.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 41.5 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
ERROR: pip's dependency resolver does not currently take into

ImportError: cannot import name 'NoExtraItems' from 'typing_extensions' (/opt/anaconda3/lib/python3.12/site-packages/typing_extensions.py)

In [2]:
# Load GTF file containing gene and transcript locations as annotated by gencode
gtf = pd.read_feather(
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)

#gtf_transcripts only keeps transcripts labeled as protein-coding
#Keep only the MANE select transcript for each gene, i.e. a consensus representative transcript per gene

# --- Filter to protein-coding MANE-select transcripts
gtf_transcripts = gene_annotation.filter_protein_coding(gtf)
gtf_transcripts = gene_annotation.filter_to_mane_select_transcript(gtf_transcripts)

# --- Initialize transcript extractor
transcript_extractor = transcript_utils.TranscriptExtractor(gtf_transcripts)

#Get interval containing LDLR
ldlr_interval = gene_annotation.get_gene_interval(gtf, gene_symbol="LDLR")

In [3]:
#Resize interval containing LDLR to a sequence of ~100KB centered around the gene sequence of LDLR
sequence_interval = ldlr_interval.resize(dna_client.SEQUENCE_LENGTH_100KB)
# sequence_interval
# sequence_interval.width

In [4]:
ldlr_transcripts = transcript_extractor.extract(ldlr_interval)
print(f'Extracted {len(ldlr_transcripts)} transcripts in this interval.')

Extracted 1 transcripts in this interval.


In [5]:
ldlr_tx = ldlr_transcripts[0]  # MANE LDLR

In [6]:
for i, exon in enumerate(ldlr_tx.exons, start=1):
    print(f"Exon {i}: {exon.chromosome}:{exon.start}-{exon.end}")

Exon 1: chr19:11089462-11089615
Exon 2: chr19:11100222-11100345
Exon 3: chr19:11102663-11102786
Exon 4: chr19:11105219-11105600
Exon 5: chr19:11106564-11106687
Exon 6: chr19:11107391-11107514
Exon 7: chr19:11110651-11110771
Exon 8: chr19:11111513-11111639
Exon 9: chr19:11113277-11113449
Exon 10: chr19:11113534-11113762
Exon 11: chr19:11116093-11116212
Exon 12: chr19:11116858-11116998
Exon 13: chr19:11120091-11120233
Exon 14: chr19:11120369-11120522
Exon 15: chr19:11123173-11123344
Exon 16: chr19:11128007-11128085
Exon 17: chr19:11129512-11129670
Exon 18: chr19:11131280-11133820


In [17]:
# # ---Define variant scorer
# # ---CenterMaskScorer used: values outside of the mask are discarded centered around the variant of interest.
# # ---Mask width specified by "width" parameter
# # ---Readout is SPLICE_SITE_USAGE: Predicted fraction of transcripts that use each site as a splice site within specified window
# # ---Scorer computes Delta(i) = model_alt(i) - model_ref(i)
# # ---model_alt(i) is the predicted fraction of transcripts using site as splice site in variant sequence, 
# # ---model_ref(i) is the predicted fraction of transcripts using site as splice site in reference sequence
# # ---DIFF_MEAN computes the average of Delta(i) for all i within the mask, producing one number per variant

# # Quick start: https://www.alphagenomedocs.com/colabs/quick_start.html#highlighting-important-regions-with-in-silico-mutagenesis
# # Output type: https://www.alphagenomedocs.com/api/generated/alphagenome.models.dna_output.OutputType.html
# # Variant scoring: https://www.alphagenomedocs.com/variant_scoring.html

# pad = 100  # bp on each side
# all_rows = []

# splice_usage_scorer = variant_scorers.CenterMaskScorer(
#     requested_output=dna_client.OutputType.SPLICE_SITE_USAGE,
#     width=501,
#     aggregation_type=variant_scorers.AggregationType.DIFF_MEAN,
# )

# for exon_idx, exon in enumerate(ldlr_tx.exons, start=1):
#     ism_interval = genome.Interval(
#         chromosome=exon.chromosome,
#         start=max(0, exon.start - pad),
#         end=exon.end + pad,
#     )
#     variant_scores = model.score_ism_variants(
#         interval=sequence_interval,   # 100 kb context around LDLR
#         ism_interval=ism_interval,    # exon4 + flanks
#         variant_scorers=[splice_usage_scorer],
#     )

#     # Collect scalar scores + variant metadata
#     scores_list = []
#     variants_list = []
#     for vs in variant_scores:
#         out = vs[0]  # one scorer
#         vmeta = out.uns["variant"]
#         delta_scalar = float(out.X.mean())  # same aggregation you used

#         # store per-variant rows (tidy)
#         all_rows.append({
#             "exon_number": exon_idx,                   # biological exon number
#             "chrom": vmeta.chromosome,
#             "pos": vmeta.position,
#             "ref": vmeta.reference_bases,
#             "alt": vmeta.alternate_bases,
#             "delta": delta_scalar,
#             "exon_start": exon.start,
#             "exon_end": exon.end,
#         })

#         # also build inputs for sequence logo (optional per-exon plotting)
#         scores_list.append(delta_scalar)
#         variants_list.append(vmeta)

# # Write one combined CSV for all exons
# import pandas as pd
# df_all = pd.DataFrame(all_rows).sort_values(["exon_number", "pos", "alt"]).reset_index(drop=True)
# df_all.to_csv("ldlr_all_exons_ism_splicing.csv", index=False)
# print("Wrote ldlr_all_exons_ism_splicing.csv)

  0%|          | 0/36 [00:00<?, ?it/s]

  0%|          | 0/33 [00:00<?, ?it/s]

  0%|          | 0/33 [00:00<?, ?it/s]

  0%|          | 0/59 [00:00<?, ?it/s]

  0%|          | 0/33 [00:00<?, ?it/s]

...

  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/36 [00:00<?, ?it/s]

  0%|          | 0/274 [00:00<?, ?it/s]

Wrote ldlr_all_exons_ism_splicing.csv and per-exon logo PNGs (if any).


In [12]:
# ---Define variant scorer
# ---CenterMaskScorer used: values outside of the mask are discarded centered around the variant of interest.
# ---Mask width specified by "width" parameter
# ---Readout is SPLICE_SITE_USAGE: Predicted fraction of transcripts that use each site as a splice site within specified window
# ---Scorer computes Delta(i) = model_alt(i) - model_ref(i)
# ---model_alt(i) is the predicted fraction of transcripts using site as splice site in variant sequence, 
# ---model_ref(i) is the predicted fraction of transcripts using site as splice site in reference sequence
# ---DIFF_MEAN computes the average of Delta(i) for all i within the mask, producing one number per variant

# Quick start: https://www.alphagenomedocs.com/colabs/quick_start.html#highlighting-important-regions-with-in-silico-mutagenesis
# Output type: https://www.alphagenomedocs.com/api/generated/alphagenome.models.dna_output.OutputType.html
# Variant scoring: https://www.alphagenomedocs.com/variant_scoring.html

# Folder for logos
os.makedirs("ldlr_exon_seqlogos", exist_ok=True)

pad = 100
all_rows = []
splice_usage_scorer = variant_scorers.CenterMaskScorer(
    requested_output=dna_client.OutputType.SPLICE_SITE_USAGE,
    width=501,
    aggregation_type=variant_scorers.AggregationType.DIFF_MEAN,
)

for exon_idx, exon in enumerate(ldlr_tx.exons, start=1):
    print(f"\nScoring exon {exon_idx}...")
    ism_interval = genome.Interval(
        chromosome=exon.chromosome,
        start=max(0, exon.start - pad),
        end=exon.end + pad,
    )

    variant_scores = model.score_ism_variants(
        interval=sequence_interval,   # 100 kb context around LDLR
        ism_interval=ism_interval,    # exon + flanks
        variant_scorers=[splice_usage_scorer],
    )

    # --- Build scores_list and variants_list for THIS exon
    scores_list = []
    variants_list = []

    for vs in variant_scores:
        out = vs[0]
        vmeta = out.uns["variant"]
        delta_scalar = -float(out.X.mean())

        all_rows.append({
            "exon_number": exon_idx,
            "chrom": vmeta.chromosome,
            "pos": vmeta.position,
            "ref": vmeta.reference_bases,
            "alt": vmeta.alternate_bases,
            "delta": delta_scalar,
            "exon_start": exon.start,
            "exon_end": exon.end,
        })

        scores_list.append(delta_scalar)
        variants_list.append(vmeta)

    ism_result = ism_utils.ism_matrix(
        scores_list,
        variants=variants_list,
    )

    # Interval corresponding to the exon (no padding)
    exon_interval = genome.Interval(
        exon.chromosome,
        exon.start,
        exon.end,
        strand=getattr(exon, "strand", None),  # optional
        name=f"exon{exon_idx}",
    )

    # --- Plot with exon highlighted
    plot_components.plot(
        [
            plot_components.SeqLogo(
                scores=ism_result,
                scores_interval=ism_interval,
                ylabel="Δ splice usage",
            )
        ],
        interval=ism_interval,
        annotations=[
            plot_components.IntervalAnnotation(
                intervals=[exon_interval],
                alpha=0.25,                 # light shading
                labels=[f"exon {exon_idx}"],
                label_angle=0,             #label at exon center
            )
        ],
        fig_width=35,
    )

    out_png = os.path.join(
        "ldlr_exon_seqlogos",
        f"ldlr_exon{exon_idx:02d}_splice_logo.png",
    )
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved {out_png}")


Scoring exon 1...


  0%|          | 0/36 [00:00<?, ?it/s]

Saved ldlr_exon_seqlogos/ldlr_exon01_splice_logo.png

Scoring exon 2...


  0%|          | 0/33 [00:00<?, ?it/s]

Saved ldlr_exon_seqlogos/ldlr_exon02_splice_logo.png

Scoring exon 3...


...

Saved ldlr_exon_seqlogos/ldlr_exon16_splice_logo.png

Scoring exon 17...


  0%|          | 0/36 [00:00<?, ?it/s]

Saved ldlr_exon_seqlogos/ldlr_exon17_splice_logo.png

Scoring exon 18...


  0%|          | 0/274 [00:00<?, ?it/s]

Saved ldlr_exon_seqlogos/ldlr_exon18_splice_logo.png
